# 0) Gruplanmis Dataset Hazirlama (Colab)

Notebook 2 egitiminden once kopya farkindalikli ve taksonomi uyumlu dataset hazirlama akisidir.

Onerilen kullanim sirasi:
1. Parametreleri duzenleyin.
2. Guncelleme/erisim kontrolu hucresini calistirin.
3. Audit hucresini calistirin.
4. Cikti dosyalarini `guided/00_start_here.md` ile okuyun.
5. Her sey temizse runtime dataset materyalizasyonunu acin.


In [ ]:
try:
    from google.colab import userdata
except Exception:
    userdata = None

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = "" if userdata is None else str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print('[GIT] GitHub token Colab secret/env uzerinden bulundu.')
else:
    print('[GIT] GitHub token bulunamadi. Public read disinda clone/push gerekiyorsa GH_TOKEN ekleyin.')

if hf_token:
    print('[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.')
else:
    print('[HF] Hugging Face token bulunamadi. Gated model gerekiyorsa Colab secret ekleyin.')

CLONE_TARGET = Path('/content/bitirmeprojesi')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _is_repo_root_inline(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _build_repo_access_url(repo_url: str, token: str | None) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root_inline(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root_inline(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root_inline(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root_inline(child):
                return child
    return None

def _ensure_repo_root_for_update_check() -> Path | None:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, gh_token)
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    return _find_repo_root_inline()

repo_root_for_update_check = _ensure_repo_root_for_update_check()
if repo_root_for_update_check is not None:
    if str(repo_root_for_update_check) not in sys.path:
        sys.path.insert(0, str(repo_root_for_update_check))
    try:
        from scripts.colab_repo_bootstrap import probe_repo_update_status

        update_status = probe_repo_update_status(repo_root_for_update_check)
        relation = str(update_status.get('relation', 'unknown'))
        if relation == 'up_to_date':
            print('[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.')
        elif relation == 'update_available':
            print(f"[KONTROL] Ilk hucre: guncelleme var. Branch={update_status.get('branch', '')}")
        else:
            print(f"[KONTROL] Ilk hucre: guncelleme durumu okunamadi: {update_status.get('message', 'bilgi yok')}")
    except Exception as exc:
        print(f'[KONTROL] Ilk hucrede guncellik kontrolu basarisiz: {exc}')
else:
    print('[KONTROL] Ilk hucrede repo bulunamadi ve auto-clone da basarisiz oldu.')


In [ ]:
import io
import json
import os
import random
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: repo kokunu bul, bagimliliklari kur ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()
ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
NOTEBOOK_FILENAME = '0_grouped_dataset_preparation.executed.ipynb'
REPO_RUN_DIR = ROOT / 'runs' / RUN_ID
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_data_prep'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_data_prep'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='0_grouped_dataset_preparation.ipynb',
    run_id=RUN_ID,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR)
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

TELEMETRY.configure_repo_output_export(
    output_dir=REPO_OUTPUT_DIR,
    notebook_filename=NOTEBOOK_FILENAME,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # REPO_DATASET_ROOT: repo icindeki class-root dataset parent koklerinden biri ya da dogrudan bir dataset klasoru olabilir.
    REPO_DATASET_ROOT = "data/class_root_dataset"

    # REPO_DATASET_NAME: REPO_DATASET_ROOT bir parent ise altindaki dataset klasor adi; REPO_DATASET_ROOT dogrudan dataset ise bos birakilabilir.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: opsiyonel repo-ici relatif dataset yolu. Bos ise REPO_DATASET_ROOT altindaki secim kullanilir.
    DATASET_ROOT = ""

    # CROP_NAME: bos ise dataset seciminden sonra notebook sorar; varsa dogrudan kullanilir.
    CROP_NAME = ""

    # PART_NAME: bos ise dataset seciminden sonra notebook sorar; bos gecerseniz unspecified kullanilir.
    PART_NAME = ""

    # PREP_ARTIFACT_ROOT: audit manifestleri ve review raporlarinin yazilacagi yer.
    PREP_ARTIFACT_ROOT = "outputs/colab_notebook_data_prep/artifacts"

    # PREPARED_RUNTIME_ROOT: onaydan sonra runtime dataset'in materyalize edilecegi kok.
    PREPARED_RUNTIME_ROOT = "data/prepared_runtime_datasets"

    # OOD_DATASET_ROOT: repo icindeki secilebilir OOD dataset klasorlerini tutan kok. OOD_DATASET_NAME verilirse materyalizasyon sirasinda kullanilir.
    OOD_DATASET_ROOT = "data/ood_dataset"

    # OOD_DATASET_NAME: opsiyonel repo OOD dataset klasor adi. Bos ise Notebook 0 OOD kopyalamaz.
    OOD_DATASET_NAME = ""

    # OOD_ROOT: opsiyonel dogrudan OOD klasor yolu. Verilirse OOD_DATASET_NAME yerine kullanilir.
    OOD_ROOT = ""

    # PREPARED_CLASS_ROOT: audit raporlarindan uretilen temiz class-root kopyasinin yazilacagi kok.
    PREPARED_CLASS_ROOT = "data/prepared_class_root_datasets"

    # PREPARE_DATASET_FROM_REPORTS: audit raporlarina gore calisma kopyasi hazirlar; True iken sonraki hucrede otomatik ilerler.
    PREPARE_DATASET_FROM_REPORTS = True

    # MATERIALIZE_AFTER_REVIEW: temiz audit/prep sonrasinda runtime datasetini otomatik yazar; sadece audit istiyorsaniz False yapin.
    MATERIALIZE_AFTER_REVIEW = True

    # CLEANUP_SEED: rapor-tabanli temizleme secimlerini deterministik tutar.
    CLEANUP_SEED = 42

    # DEVICE: DINOv3 ve BioCLIP embedding cikarimi icin cihaz.
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # EMBEDDING_BATCH_SIZE: embedding batch boyutu.
    EMBEDDING_BATCH_SIZE = 32 if DEVICE == "cuda" else 8

    # NEIGHBORS: ayni sinif aday ciftleri icin komsu sayisi.
    NEIGHBORS = 4

    PREP_DINOV3_MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    PREP_BIOCLIP_MODEL_ID = "imageomics/bioclip-2.5-vith14"

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "dataset_name": None,
        "dataset_source": None,
        "selected_ood_dataset_name": None,
        "resolved_ood_root": None,
        "prep_materialization_result": None,
        "access_report": None,
    }
    print(
        f"[PARAMS] run_id={run_id} crop={CROP_NAME or '(ask)'} part={PART_NAME or '(ask)'} "
        f"repo_dataset_root={REPO_DATASET_ROOT} repo_dataset_name={REPO_DATASET_NAME or '(ask)'} "
        f"dataset_root_override={DATASET_ROOT or '(none)'}"
    )
    print(
        f"[OOD] repo_root={OOD_DATASET_ROOT} repo_name={OOD_DATASET_NAME or '<none>'} direct_root={OOD_ROOT or '<none>'}"
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3b: Guncelleme ve Erisim Kontrolu"):
    from scripts.colab_repo_bootstrap import collect_notebook_access_report, print_notebook_access_report

    access_report = collect_notebook_access_report(
        repo_root=ROOT,
        hf_model_ids=[PREP_DINOV3_MODEL_ID, PREP_BIOCLIP_MODEL_ID],
    )
    STATE["access_report"] = access_report
    print_notebook_access_report(access_report, print_fn=print)
    TELEMETRY.update_latest(
        {
            "phase": "access_checked",
            "github_read_access": access_report.get("github", {}).get("read_access_mode"),
            "repo_update_relation": access_report.get("repo_updates", {}).get("relation"),
            "hf_access_mode": access_report.get("huggingface", {}).get("access_mode"),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.colab_dataset_layout import resolve_repo_dataset_directory
    from scripts.prepare_grouped_runtime_dataset import build_grouped_dataset_plan

    def _resolve_repo_dataset_root(repo_relative_root: str) -> Path:
        raw_repo_relative_root = str(repo_relative_root or "").strip()
        if not raw_repo_relative_root:
            raise RuntimeError("REPO_DATASET_ROOT bos birakilamaz.")
        candidate = Path(raw_repo_relative_root).expanduser()
        if candidate.is_absolute():
            raise RuntimeError("REPO_DATASET_ROOT repo-ici relatif bir yol olmali.")
        resolved = (ROOT / candidate).resolve()
        try:
            resolved.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"REPO_DATASET_ROOT repo disina cikamaz: {raw_repo_relative_root}") from exc
        return resolved

    def _normalize_token(value: str) -> str:
        normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(value or "").strip())
        while "__" in normalized:
            normalized = normalized.replace("__", "_")
        return normalized.strip("_")

    def _infer_crop_and_part_from_dataset_name(dataset_name: str) -> tuple[str, str]:
        tokens = [token for token in _normalize_token(dataset_name).split("_") if token]
        if not tokens:
            return "", "unspecified"
        if len(tokens) == 1:
            return tokens[0], "unspecified"
        return tokens[0], "_".join(tokens[1:]) or "unspecified"

    def _prompt_text(prompt: str, default_value: str = "") -> str:
        raw = str(input(prompt)).strip()
        return raw or str(default_value or "").strip()

    explicit_dataset_root = str(DATASET_ROOT).strip()
    if explicit_dataset_root:
        dataset_root = _resolve_repo_dataset_root(explicit_dataset_root)
        dataset_name = dataset_root.name
    else:
        dataset_name, dataset_root, available_dataset_names = resolve_repo_dataset_directory(
            repo_root=ROOT,
            repo_relative_root=REPO_DATASET_ROOT,
            requested_name=REPO_DATASET_NAME,
            prompt_label="class-root dataset",
        )
        print(f"[PREP] repo dataset options={available_dataset_names}")
    dataset_source = "repo"
    if not dataset_root.is_dir():
        raise RuntimeError(f"Dataset root bulunamadi: {dataset_root}")

    inferred_crop_name, inferred_part_name = _infer_crop_and_part_from_dataset_name(dataset_name)
    resolved_crop_name = str(CROP_NAME).strip()
    if not resolved_crop_name:
        resolved_crop_name = _prompt_text(
            f"CROP_NAME bos. '{dataset_name}' dataseti icin crop anahtarini girin [{inferred_crop_name or 'crop'}]: ",
            inferred_crop_name or "crop",
        )
    resolved_part_name = str(PART_NAME).strip()
    if not resolved_part_name:
        resolved_part_name = _prompt_text(
            f"PART_NAME bos. '{dataset_name}' dataseti icin part adini girin [{inferred_part_name or 'unspecified'}]: ",
            inferred_part_name or "unspecified",
        ) or "unspecified"
    CROP_NAME = resolved_crop_name
    PART_NAME = resolved_part_name
    print(f"[PREP] secilen crop={CROP_NAME} part={PART_NAME}")

    artifact_root = Path(PREP_ARTIFACT_ROOT).expanduser()
    if not artifact_root.is_absolute():
        artifact_root = (ROOT / artifact_root).resolve()

    summary = build_grouped_dataset_plan(
        class_root=dataset_root,
        crop_name=CROP_NAME,
        artifact_root=artifact_root,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
        dino_model_id=PREP_DINOV3_MODEL_ID,
        bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
        device=DEVICE,
        batch_size=EMBEDDING_BATCH_SIZE,
        neighbors=NEIGHBORS,
        progress_fn=lambda message: print(f"[PREP] {message}"),
    )

    STATE["validated"] = True
    STATE["audit_summary"] = summary
    STATE["artifact_root"] = artifact_root
    STATE["dataset_root"] = dataset_root
    STATE["dataset_name"] = dataset_name
    STATE["dataset_source"] = dataset_source
    STATE["crop_name"] = CROP_NAME
    STATE["part_name"] = PART_NAME
    print(f"[PREP] dataset_source={dataset_source} dataset_name={dataset_name} dataset_root={dataset_root}")
    print(f"[PREP] crop_name={CROP_NAME} part_name={PART_NAME}")
    print(json.dumps(summary.get("summary", {}), indent=2))
    print(f"[PREP] runtime_ready={summary.get('runtime_ready')} artifact_root={artifact_root}")
    if summary.get("blocking_issues"):
        print("[PREP] Bloklayici sorunlar:")
        for item in summary["blocking_issues"]:
            print(f"  - {item}")
    TELEMETRY.update_latest(
        {
            "phase": "data_prep_audited",
            "dataset_root": str(dataset_root),
            "dataset_name": str(dataset_name),
            "dataset_source": dataset_source,
            "artifact_root": str(artifact_root),
            "runtime_ready": bool(summary.get("runtime_ready")),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from scripts.prepare_grouped_runtime_dataset import build_prepared_dataset_key
    from scripts.prepare_materialization_dataset import prepare_class_root_for_materialization

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    summary = STATE["audit_summary"]
    if not PREPARE_DATASET_FROM_REPORTS:
        print("[PREP] PREPARE_DATASET_FROM_REPORTS=False. Audit raporlari pasif birakildi.")
    else:
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        prepared_class_root_parent = Path(PREPARED_CLASS_ROOT).expanduser()
        if not prepared_class_root_parent.is_absolute():
            prepared_class_root_parent = (ROOT / prepared_class_root_parent).resolve()
        prepared_class_root = prepared_class_root_parent / dataset_key
        prepared_artifact_root = STATE["artifact_root"].parent / f"{STATE['artifact_root'].name}_prepared"
        prep_counts = dict(summary.get("summary", {}))
        print("[PREP] Rapor tabanli hazirlik ozeti:")
        print(
            f"  dataset_key={dataset_key} total_images={prep_counts.get('total_images', 0)} "
            f"cross_class_conflicts={prep_counts.get('cross_class_conflicts', 0)} "
            f"same_class_high_risk_clusters={prep_counts.get('same_class_high_risk_clusters', 0)}"
        )
        print(f"  source_dataset_root={STATE['dataset_root']}")
        print(f"  source_artifact_root={STATE['artifact_root']}")
        print(f"  prepared_class_root={prepared_class_root}")
        print(f"  prepared_artifact_root={prepared_artifact_root}")
        print(f"  cleanup_seed={CLEANUP_SEED}")
        prep_result = prepare_class_root_for_materialization(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            audit_artifact_root=STATE["artifact_root"],
            prepared_class_root=prepared_class_root,
            prepared_artifact_root=prepared_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            dino_model_id=PREP_DINOV3_MODEL_ID,
            bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
            neighbors=NEIGHBORS,
            cleanup_seed=CLEANUP_SEED,
            quarantine_cross_class_conflicts=True,
            materialization_strategy="auto",
            progress_fn=lambda message: print(f"[PREP] {message}"),
        )
        STATE["prep_materialization_result"] = prep_result
        STATE["dataset_root"] = Path(prep_result["prepared_class_root"])
        STATE["dataset_source"] = "prepared_class_root"
        STATE["artifact_root"] = Path(prep_result["prepared_artifact_root"])
        STATE["audit_summary"] = prep_result["rerun_summary"]
        summary = STATE["audit_summary"]
        print(
            f"[PREP] prepared_runtime_ready={prep_result.get('prepared_runtime_ready')} "
            f"dataset_key={prep_result.get('dataset_key')} prepared_class_root={STATE['dataset_root']}"
        )
        print(f"[PREP] Hazirlik sonrasi artifact_root={STATE['artifact_root']}")
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_prepared",
                "dataset_root": str(STATE["dataset_root"]),
                "dataset_source": str(STATE.get("dataset_source") or "prepared_class_root"),
                "artifact_root": str(STATE["artifact_root"]),
                "runtime_ready": bool(summary.get("runtime_ready")),
            }
        )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.colab_dataset_layout import resolve_repo_dataset_directory
    from scripts.prepare_grouped_runtime_dataset import build_prepared_dataset_key, materialize_grouped_runtime_dataset

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    def _resolve_optional_ood_root() -> Path | None:
        explicit_ood_root = str(OOD_ROOT).strip()
        requested_ood_dataset = str(OOD_DATASET_NAME).strip()
        if explicit_ood_root and requested_ood_dataset:
            raise RuntimeError("OOD_ROOT ve OOD_DATASET_NAME ayni anda kullanilamaz.")
        if explicit_ood_root:
            resolved_ood_root = Path(explicit_ood_root).expanduser()
            if not resolved_ood_root.is_absolute():
                resolved_ood_root = (ROOT / resolved_ood_root).resolve()
            if not resolved_ood_root.exists():
                raise RuntimeError(f"OOD root not found: {resolved_ood_root}")
            STATE["selected_ood_dataset_name"] = None
            return resolved_ood_root
        if requested_ood_dataset:
            selected_ood_dataset_name, selected_ood_root, available_ood_dataset_names = resolve_repo_dataset_directory(
                repo_root=ROOT,
                repo_relative_root=OOD_DATASET_ROOT,
                requested_name=requested_ood_dataset,
                prompt_label="OOD dataset",
            )
            print(f"[PREP] repo OOD dataset options={available_ood_dataset_names}")
            STATE["selected_ood_dataset_name"] = selected_ood_dataset_name
            return selected_ood_root
        STATE["selected_ood_dataset_name"] = None
        return None

    summary = STATE["audit_summary"]
    if not MATERIALIZE_AFTER_REVIEW:
        print("[PREP] MATERIALIZE_AFTER_REVIEW=False. Audit dosyalari incelemeye hazir.")
    else:
        if not summary.get("runtime_ready"):
            raise RuntimeError("Audit sonucu bloklayici sorunlar iceriyor. Materyalizasyon once temizlenmeli.")
        ood_root = _resolve_optional_ood_root()
        runtime_root = Path(PREPARED_RUNTIME_ROOT).expanduser()
        if not runtime_root.is_absolute():
            runtime_root = (ROOT / runtime_root).resolve()
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        runtime_dataset_root = materialize_grouped_runtime_dataset(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            artifact_root=STATE["artifact_root"],
            runtime_root=runtime_root,
            ood_root=ood_root,
            materialization_strategy="auto",
        )
        STATE["runtime_dataset_root"] = runtime_dataset_root
        STATE["resolved_ood_root"] = str(ood_root) if ood_root is not None else ""
        print(f"[PREP] Hazir runtime dataset su klasore yazildi: {runtime_dataset_root / dataset_key}")
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_materialized",
                "runtime_dataset_root": str(runtime_dataset_root),
                "resolved_ood_root": str(STATE.get("resolved_ood_root") or ""),
                "dataset_source": str(STATE.get("dataset_source") or "repo"),
            }
        )

notebook_export_result = export_current_colab_notebook(REPO_NOTEBOOK_OUTPUT_PATH)
TELEMETRY.merge_summary_metadata(
    {
        "access_check": STATE.get("access_report", {}),
        "prep_summary": STATE.get("audit_summary", {}),
        "materialization_prep_summary": STATE.get("prep_materialization_result", {}),
        "repo_run_dir": str(REPO_RUN_DIR),
        "notebook_export_path": str(notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH),
    }
)
TELEMETRY.close(
    {
        "status": "ok",
        "runtime_ready": bool((STATE.get("audit_summary") or {}).get("runtime_ready")),
        "materialized": bool(STATE.get("runtime_dataset_root")),
        "repo_run_dir": str(REPO_RUN_DIR),
    }
)
REPO_RUN_EXPORTS = save_run_outputs_to_repo()
for key in sorted(REPO_RUN_EXPORTS):
    print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")
print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
